# 2 · Understand the two detectors

This notebook fits small teaching examples using the same code as the study. It does not change the saved models or thresholds. Run `uv run traffic-study train` to create the official fitted artifacts.

In [ ]:
import json
import pandas as pd
import numpy as np
from IPython.display import display, Image
from traffic_anomaly.paths import ROOT, RAW, RESULTS, FIGURES, config
from traffic_anomaly.data import load_split
cfg = config()
print("Project configuration found:", (ROOT / "configs/study.json").is_file())
from traffic_anomaly.models import TypicalHour, ForestDetector, calibrate
from traffic_anomaly.features import make_features
train, validation = load_split("train"), load_split("validation")

## The typical-hour baseline

For each weekday and hour, the baseline learns the median count and the median absolute deviation (MAD). The median is the middle value. MAD measures the usual distance from it. The code multiplies MAD by 1.4826 as a conventional robust spread estimate.

The score is the absolute difference from typical traffic divided by that spread. Sparse groups use broader hour-only or global statistics. A positive floor prevents division by zero.

In [ ]:
baseline = TypicalHour(cfg["baseline_min_group"], cfg["baseline_scale_floor"]).fit(train)
example = validation.head(8).copy()
centre, spread = baseline.expected(example)
example["typical_count"] = centre
example["spread"] = spread
example["baseline_score"] = baseline.score(example)
display(example)

## What the forest sees

Isolation Forest separates observations with random splits. Points separated quickly tend to look unusual. Time is supplied explicitly: sine/cosine pairs place the end of a day or week close to its start. These are clock coordinates, not future information.

The current traffic count is an input because we assess a reading after it arrives; this is not forecasting. Anomaly labels and original pre-injection values are never inputs.

In [ ]:
display(make_features(validation.head()))
forest = ForestDetector(cfg["n_estimators"], cfg["max_samples"], cfg["model_seed"]).fit(train)
validation_scores = {"Typical hour": baseline.score(validation), "Isolation Forest": forest.score(validation)}
print("Both models fitted on training observations only.")

## Convert scores into alerts

Both wrappers make larger scores mean more unusual. A threshold above about 99% of unchanged validation scores targets roughly 1% background alerts. Tied scores are handled conservatively with a strict greater-than comparison. This target is not an estimate of the proportion of faulty sensors.

In [ ]:
rows = []
for name, scores in validation_scores.items():
    threshold = calibrate(scores, cfg["primary_alert_rate"])
    rows.append({"method": name, "threshold": threshold, "observed_validation_alert_rate": (scores > threshold).mean()})
display(pd.DataFrame(rows))

## A single change

This is a learning example, not a reported test result. We multiply one validation count by 2.5 and inspect how scores change. The models remain fitted to the original training data.

In [ ]:
before = validation.iloc[[100]].copy()
after = before.copy()
after["traffic_volume"] = np.rint(after.traffic_volume * 2.5).astype(int)
display(pd.DataFrame({"version":["original","altered"], "count":[before.traffic_volume.iloc[0], after.traffic_volume.iloc[0]], "baseline_score":[baseline.score(before)[0],baseline.score(after)[0]], "forest_score":[forest.score(before)[0],forest.score(after)[0]]}))

## Check your understanding

A score is not a fault probability. The two score scales differ; we compare their alerts and ranking metrics, not raw score magnitudes. Neither model directly explains the cause of an unusual count.

Next: `03_inject_and_measure.ipynb`.